( Tratamento "atendimento_sinistros" e "engajamento_marketing" )

In [1]:
import pandas as pd
import numpy as np

# Desliga os avisos de cópia do Pandas
pd.options.mode.chained_assignment = None

print("Iniciando a Limpeza Completa e Definitiva das Tabelas...\n")

# 1. Carregando os dados brutos originais
df_sinistros = pd.read_csv('../bases/bases_kaggle/atendimento_sinistros_kaggle.csv')
df_mkt       = pd.read_csv('../bases/bases_kaggle/engajamento_marketing_kaggle.csv')

# =======================================================
# MOTOR DE LIMPEZA - ATENDIMENTO DE SINISTROS
# =======================================================
def limpar_tabela_sinistros(df_entrada):
    df = df_entrada.copy()
    
    # 1. REMOVENDO A COLUNA DE DATA DE SINISTRO
    df.drop(columns=['data_ultimo_sinistro'], errors='ignore', inplace=True)
    
    # Padronização do ID
    df.rename(columns={df.columns[0]: 'cod_individuo'}, inplace=True)
    df['cod_individuo'] = df['cod_individuo'].astype(str).str.replace('IND-', '', regex=False).str.replace('.0', '', regex=False).str.strip()
    
    # Escudo de proteção para textos (a data já não está mais aqui)
    colunas_texto = ['canal_preferencial_contato']
    colunas_protegidas = ['cod_individuo'] + colunas_texto
    col_para_matematica = [c for c in df.columns if c not in colunas_protegidas]
    
    # Conversão Numérica
    for col in col_para_matematica:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # Tratamento de Outliers e Nulos Matemáticos
    col_numericas_reais = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    for col in col_numericas_reais:
        # Capping com IQR
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        limite_inf, limite_sup = Q1 - 1.5 * (Q3 - Q1), Q3 + 1.5 * (Q3 - Q1)
        df[col] = np.clip(df[col], limite_inf, limite_sup)
        
        # Preenchimento de Nulos
        df[col] = df[col].fillna(df[col].median())
        
        # 2. RESOLVENDO O PROBLEMA DO "8.999999" (Arredondamento para 2 casas)
        df[col] = df[col].round(2)
        
    # Faxina Semântica de Textos (Canal de Contato)
    df['canal_preferencial_contato'] = df['canal_preferencial_contato'].astype(str).str.strip()
    lixos = ['#N/D', 'N/A', 'NA', '-', '?', '', 'Desconhecido', 'null', 'NULL', 'nan']
    df['canal_preferencial_contato'] = df['canal_preferencial_contato'].replace(lixos, 'Nao Informado')
    
    return df


# =======================================================
# MOTOR DE LIMPEZA - ENGAJAMENTO DE MARKETING
# =======================================================
def limpar_tabela_marketing(df_entrada):
    df = df_entrada.copy()
    
    # Padronização do ID
    df.rename(columns={df.columns[0]: 'cod_individuo'}, inplace=True)
    df['cod_individuo'] = df['cod_individuo'].astype(str).str.replace('IND-', '', regex=False).str.replace('.0', '', regex=False).str.strip()
    
    # Escudo de proteção para textos
    colunas_texto = ['tipo_veiculo', 'segmento_marketing', 'regiao_vendas', 'cluster_sugerido_crm']
    colunas_protegidas = ['cod_individuo'] + colunas_texto
    col_para_matematica = [c for c in df.columns if c not in colunas_protegidas]
    
    # Conversão Numérica
    for col in col_para_matematica:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # Tratamento de Outliers e Nulos Matemáticos
    col_numericas_reais = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    for col in col_numericas_reais:
        # Capping com IQR
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        limite_inf, limite_sup = Q1 - 1.5 * (Q3 - Q1), Q3 + 1.5 * (Q3 - Q1)
        df[col] = np.clip(df[col], limite_inf, limite_sup)
        
        # Preenchimento de Nulos
        df[col] = df[col].fillna(df[col].median())
        
        # Arredondamento padronizado também para a tabela de Marketing
        df[col] = df[col].round(2)
        
    # Faxina Semântica de Textos (Veículos e Clusters)
    for col in colunas_texto:
        df[col] = df[col].astype(str).str.strip()
        
    lixos = ['#N/D', 'N/A', 'NA', '-', '?', '', 'Desconhecido', 'null', 'NULL', 'nan']
    df['tipo_veiculo'] = df['tipo_veiculo'].replace(lixos, 'NaN')
    df['cluster_sugerido_crm'] = df['cluster_sugerido_crm'].replace(lixos, 'NaN')
    
    for col in ['segmento_marketing', 'regiao_vendas']:
        df[col] = df[col].replace('nan', 'NaN')
        
    return df

# =======================================================
# EXECUÇÃO E EXPORTAÇÃO
# =======================================================
# Roda as funções
df_sinistros_final = limpar_tabela_sinistros(df_sinistros)
df_mkt_final       = limpar_tabela_marketing(df_mkt)

# Salva as versões finais para a equipe
df_sinistros_final.to_csv('../bases/bases_kaggle/tratadas/Atendimento_Sinistros_Tratado_kaggle.csv', index=False)
df_mkt_final.to_csv('../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv', index=False)

print("SUCESSO! Suas tabelas foram 100% tratadas e salvas.")
print("-> Atendimento_Sinistros_Tratado_kaggle.csv")
print("-> Engajamento_Marketing_Tratado_kaggle.csv")

Iniciando a Limpeza Completa e Definitiva das Tabelas...

SUCESSO! Suas tabelas foram 100% tratadas e salvas.
-> Atendimento_Sinistros_Tratado_kaggle.csv
-> Engajamento_Marketing_Tratado_kaggle.csv


In [2]:
import pandas as pd
import numpy as np

print("Limpando sujeiras na coluna 'regiao_vendas'...\n")

# A sua lista de lixos padrão
lixos = ['#N/D', 'N/A', 'NA', '-', '?', '', 'Desconhecido', 'null', 'NULL', 'nan']

# Tirando os espaços em branco que podem estar escondidos e substituindo os lixos
df_mkt_final['regiao_vendas'] = df_mkt_final['regiao_vendas'].astype(str).str.strip()
df_mkt_final['regiao_vendas'] = df_mkt_final['regiao_vendas'].replace(lixos, 'NaN')

# Vamos dar uma espiada para ver as regiões que sobraram
print("Valores únicos que ficaram na coluna:")
print(df_mkt_final['regiao_vendas'].unique())

Limpando sujeiras na coluna 'regiao_vendas'...

Valores únicos que ficaram na coluna:
<StringArray>
[         'NaN',      'Sudeste',          'Sul',       'Centro',
     'Nordeste',        'Oeste', 'Centro-Oeste', 'Regiao Oeste',
            nan]
Length: 9, dtype: str


In [ ]:
import pandas as pd
import numpy as np

print("Iniciando limpeza e transformação de 'segmento_marketing'...\n")

# A nossa clássica lista de sujeiras
lixos = ['#N/D', 'N/A', 'NA', '-', '?', '', 'Desconhecido', 'null', 'NULL', 'nan']

# Trava de segurança para evitar o erro de coluna não encontrada
if 'segmento_marketing' in df_mkt_final.columns:
    
    # 1. FAXINA SEMÂNTICA
    df_mkt_final['segmento_marketing'] = df_mkt_final['segmento_marketing'].astype(str).str.strip()
    df_mkt_final['segmento_marketing'] = df_mkt_final['segmento_marketing'].replace(lixos, 'NaN')
    
    print("✅ Sujeiras removidas! Valores únicos que sobraram no segmento:")
    print(df_mkt_final['segmento_marketing'].unique(), "\n")
    
    # 2. FATIAMENTO BINÁRIO (ONE-HOT ENCODING)
    df_mkt_final = pd.get_dummies(df_mkt_final, columns=['segmento_marketing'], prefix='segmento', dtype=int)
    print("✅ Coluna transformada em binários com sucesso!")

else:
    print("⚠️ A coluna 'segmento_marketing' não foi encontrada. Você precisa rodar o Motor de Limpeza de novo para ela reaparecer!")

# 3. EXIBIÇÃO E SALVAMENTO
colunas_segmento = [col for col in df_mkt_final.columns if 'segmento_' in col]

if colunas_segmento:
    print("\nNovas colunas geradas:")
    print(colunas_segmento, "\n")
    
    # Atualiza o arquivo final
    df_mkt_final.to_csv('../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv', index=False)
    print("💾 Arquivo '../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv' atualizado e salvo no disco!\n")
    
    # Mostra o resultado final
    display(df_mkt_final[colunas_segmento].head())

Iniciando limpeza e transformação de 'segmento_marketing'...

✅ Sujeiras removidas! Valores únicos que sobraram no segmento:
<StringArray>
['Prata', 'Ouro', 'Bronze', 'Diamante', 'NaN', nan]
Length: 6, dtype: str 

✅ Coluna transformada em binários com sucesso!

Novas colunas geradas:
['segmento_Bronze', 'segmento_Diamante', 'segmento_NaN', 'segmento_Ouro', 'segmento_Prata'] 

💾 Arquivo '../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv' atualizado e salvo no disco!



,segmento_Bronze,segmento_Diamante,segmento_NaN,segmento_Ouro,segmento_Prata
0,0,0,0,0,1
1,0,0,0,1,0
2,0,0,0,0,1
3,0,0,0,0,1
4,0,0,0,0,1


Colocando Strings em Binários

In [4]:
import pandas as pd

print("Aplicando One-Hot Encoding na tabela 'df_mkt_final'...\n")

# Usa exatamente a variável que o seu código anterior gerou
df_mkt_final = pd.get_dummies(df_mkt_final, columns=['tipo_veiculo'], prefix='veiculo', dtype=int)

# Puxa o nome das novas colunas para você conferir
colunas_veiculos = [col for col in df_mkt_final.columns if 'veiculo_' in col]

print("✅ A coluna original 'tipo_veiculo' foi transformada com sucesso!")
print("Novas colunas binárias criadas:")
print(colunas_veiculos, "\n")

# Atualiza o arquivo CSV para a equipe não perder essa nova formatação
df_mkt_final.to_csv('../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv', index=False)
print("💾 Arquivo '../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv' salvo com as colunas atualizadas!\n")

# Exibe as primeiras linhas pra você ver a mágica do 1 e 0 acontecendo
display(df_mkt_final[colunas_veiculos].head())

Aplicando One-Hot Encoding na tabela 'df_mkt_final'...

✅ A coluna original 'tipo_veiculo' foi transformada com sucesso!
Novas colunas binárias criadas:
['veiculo_Carro', 'veiculo_Moto', 'veiculo_NaN', 'veiculo_Pickup', 'veiculo_SUV', 'veiculo_Van'] 

💾 Arquivo '../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv' salvo com as colunas atualizadas!



,veiculo_Carro,veiculo_Moto,veiculo_NaN,veiculo_Pickup,veiculo_SUV,veiculo_Van
0,1,0,0,0,0,0
1,0,0,0,0,1,0
2,0,0,0,0,1,0
3,1,0,0,0,0,0
4,1,0,0,0,0,0


In [5]:
import pandas as pd

print("Aplicando One-Hot Encoding na coluna 'regiao_vendas'...\n")

# Transforma as regiões em colunas binárias (0 e 1)
df_mkt_final = pd.get_dummies(df_mkt_final, columns=['regiao_vendas'], prefix='regiao', dtype=int)

# Puxa o nome das novas colunas geradas para você e o Cauê conferirem
colunas_regiao = [col for col in df_mkt_final.columns if 'regiao_' in col]

print("✅ A coluna original 'regiao_vendas' foi transformada com sucesso!")
print("Novas colunas binárias criadas:")
print(colunas_regiao, "\n")

# Atualiza o arquivo CSV final mais uma vez com essas novas colunas
df_mkt_final.to_csv('../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv', index=False)
print("💾 Arquivo '../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv' atualizado e salvo no disco!\n")

# Exibe as primeiras linhas pra você ver as novas colunas
display(df_mkt_final[colunas_regiao].head())

Aplicando One-Hot Encoding na coluna 'regiao_vendas'...

✅ A coluna original 'regiao_vendas' foi transformada com sucesso!
Novas colunas binárias criadas:
['regiao_Centro', 'regiao_Centro-Oeste', 'regiao_NaN', 'regiao_Nordeste', 'regiao_Oeste', 'regiao_Regiao Oeste', 'regiao_Sudeste', 'regiao_Sul'] 

💾 Arquivo '../bases/bases_kaggle/tratadas/Engajamento_Marketing_Tratado_kaggle.csv' atualizado e salvo no disco!



,regiao_Centro,regiao_Centro-Oeste,regiao_NaN,regiao_Nordeste,regiao_Oeste,regiao_Regiao Oeste,regiao_Sudeste,regiao_Sul
0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,1,0
2,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,1
4,1,0,0,0,0,0,0,0


In [6]:
import pandas as pd

print("Transformando 'canal_preferencial_contato' em binários na tabela de Sinistros...\n")

# Trava de segurança para evitar o KeyError
if 'canal_preferencial_contato' in df_sinistros_final.columns:
    
    # Aplica o get_dummies direto na tabela de sinistros
    df_sinistros_final = pd.get_dummies(df_sinistros_final, columns=['canal_preferencial_contato'], prefix='canal', dtype=int)
    print("✅ Sucesso! As colunas binárias de canal de contato foram criadas.")

else:
    print("⚠️ A coluna 'canal_preferencial_contato' não foi encontrada. Ela já virou binário!")

# Puxa o nome das novas colunas para conferir
colunas_canal = [col for col in df_sinistros_final.columns if 'canal_' in col]

if colunas_canal:
    print("\nNovas colunas geradas:")
    print(colunas_canal, "\n")
    
    # Atualiza o arquivo final de Sinistros
    df_sinistros_final.to_csv('../bases/bases_kaggle/tratadas/Atendimento_Sinistros_Tratado_kaggle.csv', index=False)
    print("💾 Arquivo '../bases/bases_kaggle/tratadas/Atendimento_Sinistros_Tratado_kaggle.csv' atualizado e salvo no disco!\n")
    
    # Mostra as primeiras linhas com o 1 e 0
    display(df_sinistros_final[colunas_canal].head())

Transformando 'canal_preferencial_contato' em binários na tabela de Sinistros...

✅ Sucesso! As colunas binárias de canal de contato foram criadas.

Novas colunas geradas:
['canal_Carta', 'canal_Email', 'canal_Nao Informado', 'canal_Telefone', 'canal_WhatsApp'] 

💾 Arquivo '../bases/bases_kaggle/tratadas/Atendimento_Sinistros_Tratado_kaggle.csv' atualizado e salvo no disco!



,canal_Carta,canal_Email,canal_Nao Informado,canal_Telefone,canal_WhatsApp
0,0,0,0,0,1
1,0,0,0,0,1
2,0,1,0,0,0
3,0,0,0,0,1
4,0,0,0,0,1
